# Construção provisória da coorte para a EDA 02

## Objetivo

Construir uma coorte provisória com dados reais do dataset `dados-v2`
para permitir o desenvolvimento paralelo da EDA temporal e do pipeline
de clustering.

Este notebook é um artefato temporário. Ele não substitui a coorte
oficial do projeto e poderá ser removido ou absorvido quando a construção
definitiva da coorte for entregue.

## Critérios de elegibilidade aplicados

- hospitalização confirmada, com `hospital = 1`;
- `dt_interna` válida;
- internação entre 01/01/2019 e 28/09/2026;
- evolução elegível, com `evolucao` igual a 1 ou 2;
- remoção de linhas integralmente duplicadas.

## Tratamentos que não serão realizados arbitrariamente

- registros diferentes com o mesmo `nu_notific` não serão eliminados;
- intervalos negativos entre sintomas e internação serão sinalizados,
  mas não excluídos automaticamente;
- ausência em variáveis preditoras não causará exclusão;
- `dt_notific` ou `dt_sin_pri` fora do período da coorte serão
  sinalizadas, mas não excluirão o registro quando `dt_interna` for
  elegível;
- ausência não será convertida automaticamente em resposta negativa.

## Saídas

- `data/analytical/coorte_provisoria_b.parquet`;
- `outputs/tables/coorte_provisoria_b_waterfall.csv`;
- `outputs/tables/coorte_provisoria_b_auditoria_duplicidades.csv`;
- `outputs/tables/coorte_provisoria_b_auditoria_temporal.csv`;
- `outputs/tables/coorte_provisoria_b_resumo.csv`.

In [ ]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    """
    Localiza a raiz do projeto procurando pelas pastas src e configs.
    """

    pasta_atual = Path.cwd().resolve()

    pastas_candidatas = [
        pasta_atual
    ] + list(
        pasta_atual.parents
    )

    for pasta in pastas_candidatas:
        possui_src = (
            pasta
            / "src"
        ).is_dir()

        possui_configs = (
            pasta
            / "configs"
        ).is_dir()

        if possui_src and possui_configs:
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ROOT_DIR = encontrar_raiz_projeto()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT_DIR),
    )

from src.utils import config as C
from src.utils import env
from src.utils import storage

env.carregar_dotenv()
env.garantir_diretorios()

DATA_INICIAL_COORTE = pd.Timestamp(
    "2019-01-01"
)

DATA_FINAL_COORTE = pd.Timestamp(
    "2026-09-28"
)

ANO_MINIMO_CONVERSAO = 1900
ANO_MAXIMO_CONVERSAO = 2026

CAMINHO_DICIONARIO = (
    C.CONFIGS_DIR
    / "dicionario_variaveis_v1.csv"
)

CAMINHO_COORTE_PROVISORIA = (
    C.ANALYTICAL_DIR
    / "coorte_provisoria_b.parquet"
)

CAMINHO_WATERFALL = (
    C.TABLES_DIR
    / "coorte_provisoria_b_waterfall.csv"
)

CAMINHO_AUDITORIA_DUPLICIDADES = (
    C.TABLES_DIR
    / "coorte_provisoria_b_auditoria_duplicidades.csv"
)

CAMINHO_AUDITORIA_TEMPORAL = (
    C.TABLES_DIR
    / "coorte_provisoria_b_auditoria_temporal.csv"
)

CAMINHO_RESUMO = (
    C.TABLES_DIR
    / "coorte_provisoria_b_resumo.csv"
)

print(
    "Raiz do projeto:",
    ROOT_DIR,
)

print(
    "Versão dos dados:",
    C.DATA_VERSION,
)

print(
    "Arquivo oficial:",
    C.PARQUET_NAME,
)

print(
    "Período provisório:",
    DATA_INICIAL_COORTE.date(),
    "a",
    DATA_FINAL_COORTE.date(),
)

print(
    "Saída da coorte:",
    CAMINHO_COORTE_PROVISORIA,
)

In [ ]:
if not CAMINHO_DICIONARIO.exists():
    raise FileNotFoundError(
        "O dicionário operacional não foi encontrado em: "
        f"{CAMINHO_DICIONARIO}"
    )

dicionario_variaveis = pd.read_csv(
    CAMINHO_DICIONARIO,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)

if "coluna" not in dicionario_variaveis.columns:
    raise KeyError(
        "O dicionário operacional não contém a coluna 'coluna'."
    )

colunas_dicionario = (
    dicionario_variaveis[
        "coluna"
    ]
    .astype("string")
    .str.strip()
)

colunas_dicionario = (
    colunas_dicionario.loc[
        colunas_dicionario.ne("")
    ]
    .drop_duplicates()
    .tolist()
)

COLUNAS_ADICIONAIS = [
    "id_registro",
    "ano_arquivo",
    "data_snapshot",
    "arquivo_origem",
    "nosocomial",
]

COLUNAS_SOLICITADAS = list(
    dict.fromkeys(
        COLUNAS_ADICIONAIS
        + colunas_dicionario
    )
)

caminho_parquet = storage.get_parquet(
    force=False
)

if not caminho_parquet.exists():
    raise FileNotFoundError(
        "O Parquet oficial não foi encontrado em: "
        f"{caminho_parquet}"
    )

arquivo_parquet = pq.ParquetFile(
    caminho_parquet
)

colunas_disponiveis = set(
    arquivo_parquet.schema_arrow.names
)

COLUNAS_EXISTENTES = [
    coluna
    for coluna in COLUNAS_SOLICITADAS
    if coluna in colunas_disponiveis
]

COLUNAS_AUSENTES = [
    coluna
    for coluna in COLUNAS_SOLICITADAS
    if coluna not in colunas_disponiveis
]

COLUNAS_MINIMAS = {
    "nu_notific",
    "hospital",
    "dt_notific",
    "dt_sin_pri",
    "dt_interna",
    "evolucao",
}

colunas_minimas_ausentes = (
    COLUNAS_MINIMAS
    - set(
        COLUNAS_EXISTENTES
    )
)

if colunas_minimas_ausentes:
    raise KeyError(
        "O Parquet não contém as colunas mínimas: "
        f"{sorted(colunas_minimas_ausentes)}"
    )

print(
    "Colunas solicitadas:",
    len(
        COLUNAS_SOLICITADAS
    ),
)

print(
    "Colunas encontradas:",
    len(
        COLUNAS_EXISTENTES
    ),
)

print(
    "Colunas opcionais ausentes:",
    len(
        COLUNAS_AUSENTES
    ),
)

if COLUNAS_AUSENTES:
    display(
        pd.DataFrame(
            {
                "coluna_opcional_ausente": (
                    COLUNAS_AUSENTES
                ),
            }
        )
    )

print(
    "nosocomial disponível:",
    "nosocomial" in COLUNAS_EXISTENTES,
)

In [ ]:
def normalizar_codigo(
    serie: pd.Series,
) -> pd.Series:
    """
    Normaliza códigos textuais sem interpretar ausência como resposta.
    """

    resultado = (
        serie.astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "NaN": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            }
        )
    )

    return resultado


def converter_data_mista_segura(
    serie: pd.Series,
    ano_minimo: int = ANO_MINIMO_CONVERSAO,
    ano_maximo: int = ANO_MAXIMO_CONVERSAO,
) -> pd.Series:
    """
    Converte datas nos formatos AAAA-MM-DD e DD/MM/AAAA.

    Datas não reconhecidas, calendários inválidos e anos fora da
    faixa configurada tornam-se NaT.
    """

    texto = (
        serie.astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "NaN": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            }
        )
    )

    resultado = pd.Series(
        pd.NaT,
        index=serie.index,
        dtype="datetime64[ns]",
    )

    mascara_iso = texto.str.match(
        r"^\d{4}-\d{2}-\d{2}$",
        na=False,
    )

    mascara_br = texto.str.match(
        r"^\d{2}/\d{2}/\d{4}$",
        na=False,
    )

    ano_iso = pd.to_numeric(
        texto.str.slice(
            0,
            4,
        ),
        errors="coerce",
    )

    ano_br = pd.to_numeric(
        texto.str.slice(
            6,
            10,
        ),
        errors="coerce",
    )

    mascara_iso_valida = (
        mascara_iso
        & ano_iso.between(
            ano_minimo,
            ano_maximo,
        )
    )

    mascara_br_valida = (
        mascara_br
        & ano_br.between(
            ano_minimo,
            ano_maximo,
        )
    )

    resultado.loc[
        mascara_iso_valida
    ] = pd.to_datetime(
        texto.loc[
            mascara_iso_valida
        ],
        format="%Y-%m-%d",
        errors="coerce",
    )

    resultado.loc[
        mascara_br_valida
    ] = pd.to_datetime(
        texto.loc[
            mascara_br_valida
        ],
        format="%d/%m/%Y",
        errors="coerce",
    )

    return resultado


def adicionar_etapa_waterfall(
    etapas: list,
    etapa: str,
    registros_antes: int,
    registros_depois: int,
    motivo: str,
) -> None:
    """
    Registra uma etapa do fluxo de seleção.
    """

    registros_excluidos = (
        registros_antes
        - registros_depois
    )

    if registros_antes == 0:
        percentual_excluido = 0.0
    else:
        percentual_excluido = (
            registros_excluidos
            / registros_antes
        )

        percentual_excluido = (
            percentual_excluido
            * 100
        )

    etapas.append(
        {
            "etapa": etapa,
            "registros_antes": int(
                registros_antes
            ),
            "registros_excluidos": int(
                registros_excluidos
            ),
            "registros_depois": int(
                registros_depois
            ),
            "percentual_excluido": float(
                percentual_excluido
            ),
            "motivo": motivo,
        }
    )

In [ ]:
base_original = storage.read_srag(
    columns=COLUNAS_EXISTENTES
)

if base_original.empty:
    raise ValueError(
        "A base oficial foi carregada sem registros."
    )

quantidade_registros_originais = len(
    base_original
)

quantidade_colunas_originais = (
    base_original.shape[1]
)

memoria_base_original_bytes = (
    base_original
    .memory_usage(
        deep=True
    )
    .sum()
)

memoria_base_original_mb = (
    memoria_base_original_bytes
    / 1_048_576
)

print(
    "Dimensão carregada:",
    base_original.shape,
)

print(
    "Memória aproximada em MB:",
    round(
        memoria_base_original_mb,
        2,
    ),
)

display(
    base_original.head()
)

In [ ]:
waterfall = [
    {
        "etapa": "registros_originais",
        "registros_antes": (
            quantidade_registros_originais
        ),
        "registros_excluidos": 0,
        "registros_depois": (
            quantidade_registros_originais
        ),
        "percentual_excluido": 0.0,
        "motivo": (
            "Total de registros carregados do dataset oficial."
        ),
    }
]

n_antes_duplicidades = len(
    base_original
)

mascara_duplicidade_exata = (
    base_original.duplicated(
        keep="first"
    )
)

quantidade_duplicidades_exatas = int(
    mascara_duplicidade_exata.sum()
)

base_trabalho = (
    base_original.loc[
        ~mascara_duplicidade_exata
    ]
    .copy()
    .reset_index(drop=True)
)

adicionar_etapa_waterfall(
    etapas=waterfall,
    etapa="duplicidades_exatas",
    registros_antes=n_antes_duplicidades,
    registros_depois=len(
        base_trabalho
    ),
    motivo=(
        "Remoção somente de linhas integralmente idênticas "
        "nos campos originais carregados."
    ),
)

print(
    "Duplicidades exatas removidas:",
    quantidade_duplicidades_exatas,
)

print(
    "Registros restantes:",
    len(
        base_trabalho
    ),
)

del base_original
del mascara_duplicidade_exata

gc.collect()

In [ ]:
base_trabalho[
    "hospital_normalizado"
] = normalizar_codigo(
    base_trabalho[
        "hospital"
    ]
)

base_trabalho[
    "evolucao_normalizada"
] = normalizar_codigo(
    base_trabalho[
        "evolucao"
    ]
)

base_trabalho[
    "nu_notific_normalizado"
] = normalizar_codigo(
    base_trabalho[
        "nu_notific"
    ]
)

if "nosocomial" in base_trabalho.columns:
    base_trabalho[
        "nosocomial_normalizado"
    ] = normalizar_codigo(
        base_trabalho[
            "nosocomial"
        ]
    )
else:
    base_trabalho[
        "nosocomial_normalizado"
    ] = pd.Series(
        pd.NA,
        index=base_trabalho.index,
        dtype="string",
    )

COLUNAS_DATA_ESSENCIAIS = [
    "dt_notific",
    "dt_sin_pri",
    "dt_interna",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

for coluna in COLUNAS_DATA_ESSENCIAIS:
    if coluna in base_trabalho.columns:
        nome_convertido = (
            f"{coluna}_convertida"
        )

        base_trabalho[
            nome_convertido
        ] = converter_data_mista_segura(
            base_trabalho[
                coluna
            ]
        )

COLUNAS_CONVERTIDAS_OBRIGATORIAS = [
    "dt_notific_convertida",
    "dt_sin_pri_convertida",
    "dt_interna_convertida",
]

colunas_convertidas_ausentes = [
    coluna
    for coluna in COLUNAS_CONVERTIDAS_OBRIGATORIAS
    if coluna not in base_trabalho.columns
]

if colunas_convertidas_ausentes:
    raise KeyError(
        "As datas obrigatórias não foram convertidas: "
        f"{colunas_convertidas_ausentes}"
    )

print(
    "Códigos e datas essenciais normalizados."
)

In [ ]:
n_antes_hospitalizacao = len(
    base_trabalho
)

base_trabalho = (
    base_trabalho.loc[
        base_trabalho[
            "hospital_normalizado"
        ].eq("1")
    ]
    .copy()
    .reset_index(drop=True)
)

adicionar_etapa_waterfall(
    etapas=waterfall,
    etapa="hospitalizacao_confirmada",
    registros_antes=n_antes_hospitalizacao,
    registros_depois=len(
        base_trabalho
    ),
    motivo=(
        "Manutenção somente de registros com hospital = 1."
    ),
)


n_antes_data_internacao = len(
    base_trabalho
)

base_trabalho = (
    base_trabalho.loc[
        base_trabalho[
            "dt_interna_convertida"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

adicionar_etapa_waterfall(
    etapas=waterfall,
    etapa="data_internacao_valida",
    registros_antes=n_antes_data_internacao,
    registros_depois=len(
        base_trabalho
    ),
    motivo=(
        "Manutenção somente de registros com dt_interna válida."
    ),
)


n_antes_periodo = len(
    base_trabalho
)

mascara_periodo = (
    base_trabalho[
        "dt_interna_convertida"
    ]
    .between(
        DATA_INICIAL_COORTE,
        DATA_FINAL_COORTE,
    )
)

base_trabalho = (
    base_trabalho.loc[
        mascara_periodo
    ]
    .copy()
    .reset_index(drop=True)
)

adicionar_etapa_waterfall(
    etapas=waterfall,
    etapa="periodo_da_coorte",
    registros_antes=n_antes_periodo,
    registros_depois=len(
        base_trabalho
    ),
    motivo=(
        "Internação entre 01/01/2019 e 28/09/2026."
    ),
)


n_antes_desfecho = len(
    base_trabalho
)

mascara_desfecho = (
    base_trabalho[
        "evolucao_normalizada"
    ]
    .isin(
        [
            "1",
            "2",
        ]
    )
)

base_trabalho = (
    base_trabalho.loc[
        mascara_desfecho
    ]
    .copy()
    .reset_index(drop=True)
)

adicionar_etapa_waterfall(
    etapas=waterfall,
    etapa="desfecho_elegivel",
    registros_antes=n_antes_desfecho,
    registros_depois=len(
        base_trabalho
    ),
    motivo=(
        "Manutenção somente de evolucao = 1 ou evolucao = 2."
    ),
)

waterfall_df = pd.DataFrame(
    waterfall
)

display(
    waterfall_df
)

gc.collect()

In [ ]:
coorte_provisoria = base_trabalho

coorte_provisoria[
    "alvo_obito_srag"
] = (
    coorte_provisoria[
        "evolucao_normalizada"
    ]
    .eq("2")
    .astype("int8")
)

coorte_provisoria[
    "ano_internacao"
] = (
    coorte_provisoria[
        "dt_interna_convertida"
    ]
    .dt.year
    .astype("Int64")
)

coorte_provisoria[
    "mes_internacao"
] = (
    coorte_provisoria[
        "dt_interna_convertida"
    ]
    .dt.month
    .astype("Int64")
)

coorte_provisoria[
    "ano_notificacao"
] = (
    coorte_provisoria[
        "dt_notific_convertida"
    ]
    .dt.year
    .astype("Int64")
)

coorte_provisoria[
    "ano_sintomas"
] = (
    coorte_provisoria[
        "dt_sin_pri_convertida"
    ]
    .dt.year
    .astype("Int64")
)

coorte_provisoria[
    "dias_sintomas_ate_internacao"
] = (
    coorte_provisoria[
        "dt_interna_convertida"
    ]
    - coorte_provisoria[
        "dt_sin_pri_convertida"
    ]
).dt.days.astype("Int64")

coorte_provisoria[
    "dias_notificacao_ate_internacao"
] = (
    coorte_provisoria[
        "dt_interna_convertida"
    ]
    - coorte_provisoria[
        "dt_notific_convertida"
    ]
).dt.days.astype("Int64")

coorte_provisoria[
    "flag_internacao_antes_sintomas"
] = (
    coorte_provisoria[
        "dias_sintomas_ate_internacao"
    ]
    .lt(0)
    .fillna(False)
    .astype("int8")
)

coorte_provisoria[
    "flag_notificacao_apos_internacao"
] = (
    coorte_provisoria[
        "dias_notificacao_ate_internacao"
    ]
    .lt(0)
    .fillna(False)
    .astype("int8")
)

coorte_provisoria[
    "flag_notificacao_fora_periodo"
] = (
    coorte_provisoria[
        "dt_notific_convertida"
    ]
    .notna()
    & ~coorte_provisoria[
        "dt_notific_convertida"
    ]
    .between(
        DATA_INICIAL_COORTE,
        DATA_FINAL_COORTE,
    )
).astype("int8")

coorte_provisoria[
    "flag_sintomas_fora_periodo"
] = (
    coorte_provisoria[
        "dt_sin_pri_convertida"
    ]
    .notna()
    & ~coorte_provisoria[
        "dt_sin_pri_convertida"
    ]
    .between(
        DATA_INICIAL_COORTE,
        DATA_FINAL_COORTE,
    )
).astype("int8")

coorte_provisoria[
    "flag_dt_sin_pri_ausente"
] = (
    coorte_provisoria[
        "dt_sin_pri_convertida"
    ]
    .isna()
    .astype("int8")
)

coorte_provisoria[
    "flag_dt_notific_ausente"
] = (
    coorte_provisoria[
        "dt_notific_convertida"
    ]
    .isna()
    .astype("int8")
)

coorte_provisoria[
    "flag_nosocomial"
] = (
    coorte_provisoria[
        "nosocomial_normalizado"
    ]
    .eq("1")
    .fillna(False)
    .astype("int8")
)

coorte_provisoria[
    "flag_intervalo_negativo_sem_nosocomial"
] = (
    coorte_provisoria[
        "flag_internacao_antes_sintomas"
    ]
    .eq(1)
    & coorte_provisoria[
        "flag_nosocomial"
    ]
    .ne(1)
).astype("int8")

coorte_provisoria[
    "periodo_incompleto"
] = (
    coorte_provisoria[
        "ano_internacao"
    ]
    .eq(2026)
    .astype("int8")
)

print(
    "Alvo, intervalos e flags temporais criados."
)

In [ ]:
intervalo = (
    coorte_provisoria[
        "dias_sintomas_ate_internacao"
    ]
)

condicoes_intervalo = [
    intervalo.isna(),
    intervalo.lt(-30),
    intervalo.between(
        -30,
        -8,
        inclusive="both",
    ),
    intervalo.between(
        -7,
        -1,
        inclusive="both",
    ),
    intervalo.eq(0),
    intervalo.between(
        1,
        7,
        inclusive="both",
    ),
    intervalo.between(
        8,
        14,
        inclusive="both",
    ),
    intervalo.between(
        15,
        30,
        inclusive="both",
    ),
    intervalo.gt(30),
]

rotulos_intervalo = [
    "ausente",
    "menor_que_menos_30",
    "menos_30_a_menos_8",
    "menos_7_a_menos_1",
    "zero",
    "1_a_7",
    "8_a_14",
    "15_a_30",
    "maior_que_30",
]

classificacao_intervalo = np.select(
    condicoes_intervalo,
    rotulos_intervalo,
    default="nao_classificado",
)

coorte_provisoria[
    "faixa_dias_sintomas_ate_internacao"
] = pd.Series(
    classificacao_intervalo,
    index=coorte_provisoria.index,
    dtype="string",
)

tabela_intervalos = (
    coorte_provisoria[
        "faixa_dias_sintomas_ate_internacao"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "faixa"
    )
    .reset_index(
        name="quantidade"
    )
)

display(
    tabela_intervalos
)

In [ ]:
contagem_nu_notific = (
    coorte_provisoria[
        "nu_notific_normalizado"
    ]
    .dropna()
    .value_counts()
)

coorte_provisoria[
    "quantidade_ocorrencias_nu_notific"
] = (
    coorte_provisoria[
        "nu_notific_normalizado"
    ]
    .map(
        contagem_nu_notific
    )
    .astype("Int64")
)

coorte_provisoria[
    "flag_nu_notific_repetido"
] = (
    coorte_provisoria[
        "quantidade_ocorrencias_nu_notific"
    ]
    .gt(1)
    .fillna(False)
    .astype("int8")
)

coorte_provisoria[
    "flag_nu_notific_ausente"
] = (
    coorte_provisoria[
        "nu_notific_normalizado"
    ]
    .isna()
    .astype("int8")
)

coorte_provisoria[
    "status_coorte"
] = "provisoria_pessoa_b"

coorte_provisoria[
    "data_corte_coorte"
] = DATA_FINAL_COORTE

print(
    "Linhas com nu_notific repetido:",
    int(
        coorte_provisoria[
            "flag_nu_notific_repetido"
        ].sum()
    ),
)

print(
    "Linhas com nu_notific ausente:",
    int(
        coorte_provisoria[
            "flag_nu_notific_ausente"
        ].sum()
    ),
)

In [ ]:
auditoria_duplicidades = (
    coorte_provisoria.loc[
        coorte_provisoria[
            "flag_nu_notific_repetido"
        ].eq(1)
    ]
    .groupby(
        "nu_notific_normalizado",
        dropna=False,
    )
    .agg(
        quantidade_ocorrencias=(
            "nu_notific_normalizado",
            "size",
        ),
        quantidade_anos_internacao=(
            "ano_internacao",
            "nunique",
        ),
        quantidade_datas_internacao=(
            "dt_interna_convertida",
            "nunique",
        ),
        quantidade_desfechos=(
            "evolucao_normalizada",
            "nunique",
        ),
        quantidade_datas_sintomas=(
            "dt_sin_pri_convertida",
            "nunique",
        ),
    )
    .reset_index()
)

auditoria_duplicidades[
    "flag_ano_internacao_divergente"
] = (
    auditoria_duplicidades[
        "quantidade_anos_internacao"
    ]
    .gt(1)
    .astype("int8")
)

auditoria_duplicidades[
    "flag_dt_interna_divergente"
] = (
    auditoria_duplicidades[
        "quantidade_datas_internacao"
    ]
    .gt(1)
    .astype("int8")
)

auditoria_duplicidades[
    "flag_evolucao_divergente"
] = (
    auditoria_duplicidades[
        "quantidade_desfechos"
    ]
    .gt(1)
    .astype("int8")
)

auditoria_duplicidades[
    "flag_dt_sin_pri_divergente"
] = (
    auditoria_duplicidades[
        "quantidade_datas_sintomas"
    ]
    .gt(1)
    .astype("int8")
)

auditoria_duplicidades = (
    auditoria_duplicidades
    .sort_values(
        [
            "quantidade_ocorrencias",
            "nu_notific_normalizado",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

auditoria_temporal = pd.DataFrame(
    {
        "verificacao": [
            "dt_sin_pri_ausente",
            "dt_notific_ausente",
            "internacao_antes_sintomas",
            "intervalo_negativo_sem_nosocomial",
            "notificacao_apos_internacao",
            "notificacao_fora_periodo",
            "sintomas_fora_periodo",
            "nu_notific_repetido",
            "nu_notific_ausente",
            "periodo_incompleto_2026",
        ],
        "quantidade": [
            int(
                coorte_provisoria[
                    "flag_dt_sin_pri_ausente"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_dt_notific_ausente"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_internacao_antes_sintomas"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_intervalo_negativo_sem_nosocomial"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_notificacao_apos_internacao"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_notificacao_fora_periodo"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_sintomas_fora_periodo"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_nu_notific_repetido"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_nu_notific_ausente"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "periodo_incompleto"
                ].sum()
            ),
        ],
    }
)

auditoria_temporal[
    "percentual_coorte"
] = (
    auditoria_temporal[
        "quantidade"
    ]
    .div(
        len(
            coorte_provisoria
        )
    )
    .mul(100)
)

display(
    auditoria_temporal
)

display(
    auditoria_duplicidades.head(20)
)

In [ ]:
distribuicao_alvo = (
    coorte_provisoria[
        "alvo_obito_srag"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

resumo_coorte = pd.DataFrame(
    {
        "metrica": [
            "versao_dados",
            "status_coorte",
            "data_inicial_coorte",
            "data_final_coorte",
            "registros_originais",
            "colunas_originais_carregadas",
            "duplicidades_exatas_removidas",
            "registros_coorte_provisoria",
            "colunas_coorte_provisoria",
            "alvo_0_cura",
            "alvo_1_obito_srag",
            "linhas_nu_notific_repetido",
            "identificadores_repetidos_distintos",
            "internacao_antes_sintomas",
            "intervalo_negativo_sem_nosocomial",
            "registros_2026",
        ],
        "valor": [
            C.DATA_VERSION,
            "provisoria_pessoa_b",
            str(
                DATA_INICIAL_COORTE.date()
            ),
            str(
                DATA_FINAL_COORTE.date()
            ),
            quantidade_registros_originais,
            quantidade_colunas_originais,
            quantidade_duplicidades_exatas,
            len(
                coorte_provisoria
            ),
            coorte_provisoria.shape[1],
            int(
                distribuicao_alvo.get(
                    0,
                    0,
                )
            ),
            int(
                distribuicao_alvo.get(
                    1,
                    0,
                )
            ),
            int(
                coorte_provisoria[
                    "flag_nu_notific_repetido"
                ].sum()
            ),
            len(
                auditoria_duplicidades
            ),
            int(
                coorte_provisoria[
                    "flag_internacao_antes_sintomas"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "flag_intervalo_negativo_sem_nosocomial"
                ].sum()
            ),
            int(
                coorte_provisoria[
                    "periodo_incompleto"
                ].sum()
            ),
        ],
    }
)

display(
    resumo_coorte
)

In [ ]:
existem_duplicidades_exatas_restantes = (
    coorte_provisoria[
        COLUNAS_EXISTENTES
    ]
    .duplicated()
    .any()
)

verificacoes_pre_salvamento = {
    "coorte_nao_vazia": (
        not coorte_provisoria.empty
    ),
    "hospitalizacao_confirmada": (
        coorte_provisoria[
            "hospital_normalizado"
        ]
        .eq("1")
        .all()
    ),
    "dt_interna_valida": (
        coorte_provisoria[
            "dt_interna_convertida"
        ]
        .notna()
        .all()
    ),
    "periodo_correto": (
        coorte_provisoria[
            "dt_interna_convertida"
        ]
        .between(
            DATA_INICIAL_COORTE,
            DATA_FINAL_COORTE,
        )
        .all()
    ),
    "desfecho_elegivel": (
        coorte_provisoria[
            "evolucao_normalizada"
        ]
        .isin(
            [
                "1",
                "2",
            ]
        )
        .all()
    ),
    "alvo_binario": (
        set(
            coorte_provisoria[
                "alvo_obito_srag"
            ]
            .dropna()
            .unique()
        )
        .issubset(
            {
                0,
                1,
            }
        )
    ),
    "sem_duplicidades_exatas_restantes": (
        not existem_duplicidades_exatas_restantes
    ),
    "flag_repeticao_identificador_criada": (
        "flag_nu_notific_repetido"
        in coorte_provisoria.columns
    ),
    "waterfall_criado": (
        not waterfall_df.empty
    ),
    "auditoria_temporal_criada": (
        not auditoria_temporal.empty
    ),
}

verificacoes_pre_salvamento_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_pre_salvamento.keys()
        ),
        "resultado": list(
            verificacoes_pre_salvamento.values()
        ),
    }
)

display(
    verificacoes_pre_salvamento_df
)

if not all(
    verificacoes_pre_salvamento.values()
):
    falhas = [
        verificacao
        for verificacao, resultado
        in verificacoes_pre_salvamento.items()
        if not resultado
    ]

    raise RuntimeError(
        "A coorte provisória não pode ser salva. "
        f"Verificações com falha: {falhas}"
    )

print(
    "Validações anteriores ao salvamento concluídas."
)

In [ ]:
C.ANALYTICAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

C.TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

coorte_provisoria.to_parquet(
    CAMINHO_COORTE_PROVISORIA,
    index=False,
    compression="zstd",
)

waterfall_df.to_csv(
    CAMINHO_WATERFALL,
    index=False,
    encoding="utf-8-sig",
)

auditoria_duplicidades.to_csv(
    CAMINHO_AUDITORIA_DUPLICIDADES,
    index=False,
    encoding="utf-8-sig",
)

auditoria_temporal.to_csv(
    CAMINHO_AUDITORIA_TEMPORAL,
    index=False,
    encoding="utf-8-sig",
)

resumo_coorte.to_csv(
    CAMINHO_RESUMO,
    index=False,
    encoding="utf-8-sig",
)

coorte_reaberta = pd.read_parquet(
    CAMINHO_COORTE_PROVISORIA
)

verificacoes_finais = {
    "arquivo_parquet_criado": (
        CAMINHO_COORTE_PROVISORIA.exists()
    ),
    "arquivo_waterfall_criado": (
        CAMINHO_WATERFALL.exists()
    ),
    "arquivo_auditoria_duplicidades_criado": (
        CAMINHO_AUDITORIA_DUPLICIDADES.exists()
    ),
    "arquivo_auditoria_temporal_criado": (
        CAMINHO_AUDITORIA_TEMPORAL.exists()
    ),
    "arquivo_resumo_criado": (
        CAMINHO_RESUMO.exists()
    ),
    "parquet_reaberto_nao_vazio": (
        not coorte_reaberta.empty
    ),
    "mesmo_numero_de_linhas": (
        len(
            coorte_reaberta
        )
        == len(
            coorte_provisoria
        )
    ),
    "mesmo_numero_de_colunas": (
        coorte_reaberta.shape[1]
        == coorte_provisoria.shape[1]
    ),
    "status_provisorio_preservado": (
        coorte_reaberta[
            "status_coorte"
        ]
        .eq(
            "provisoria_pessoa_b"
        )
        .all()
    ),
    "alvo_preservado": (
        coorte_reaberta[
            "alvo_obito_srag"
        ]
        .isin(
            [
                0,
                1,
            ]
        )
        .all()
    ),
}

verificacoes_finais_df = pd.DataFrame(
    {
        "verificacao": list(
            verificacoes_finais.keys()
        ),
        "resultado": list(
            verificacoes_finais.values()
        ),
    }
)

display(
    verificacoes_finais_df
)

if not all(
    verificacoes_finais.values()
):
    falhas = [
        verificacao
        for verificacao, resultado
        in verificacoes_finais.items()
        if not resultado
    ]

    raise RuntimeError(
        "A construção provisória terminou com falhas: "
        f"{falhas}"
    )

tamanho_parquet_mb = (
    CAMINHO_COORTE_PROVISORIA
    .stat()
    .st_size
    / 1_048_576
)

print(
    "Coorte provisória construída e validada com sucesso."
)

print(
    "Arquivo:",
    CAMINHO_COORTE_PROVISORIA,
)

print(
    "Linhas:",
    len(
        coorte_reaberta
    ),
)

print(
    "Colunas:",
    coorte_reaberta.shape[1],
)

print(
    "Tamanho em MB:",
    round(
        tamanho_parquet_mb,
        2,
    ),
)